In [0]:
# Load US income dataset (e.g., UCI Adult dataset)
income_df = spark.read.csv("/databricks-datasets/adult/adult.data", header=False, inferSchema=True)
columns = [
    "age", "workclass", "fnlwgt", "education", "education_num", "marital_status",
    "occupation", "relationship", "race", "sex", "capital_gain", "capital_loss",
    "hours_per_week", "native_country", "income"
]
income_df = income_df.toDF(*columns)


In [0]:
display(income_df.limit(10))

In [0]:
# Preprocessing: StringIndexer for categorical columns, VectorAssembler for features
from pyspark.ml.feature import StringIndexer, VectorAssembler, OneHotEncoder
categorical_cols = ["workclass", "education", "marital_status", "occupation", "relationship", "race", "sex", "native_country"]
indexers = [StringIndexer(inputCol=col, outputCol=col+"_idx", handleInvalid="keep") for col in categorical_cols]
encoder = OneHotEncoder(inputCols=[col+"_idx" for col in categorical_cols], outputCols=[col+"_enc" for col in categorical_cols])
label_indexer = StringIndexer(inputCol="income", outputCol="label", handleInvalid="keep")
numeric_cols = ["age", "fnlwgt", "education_num", "capital_gain", "capital_loss", "hours_per_week"]
assembler = VectorAssembler(inputCols=[col+"_enc" for col in categorical_cols] + numeric_cols, outputCol="features")


In [0]:
from pyspark.ml import Pipeline
pipeline = Pipeline(stages=indexers + [encoder, label_indexer, assembler])
processed_df = pipeline.fit(income_df).transform(income_df).select("features", "label")

In [0]:
# Split data
train_df, test_df = processed_df.randomSplit([0.8, 0.2], seed=42)


In [0]:
display(train_df.limit(10))

In [0]:
%pip install xgboost>=2.0.0

In [0]:
dbutils.library.restartPython()

In [0]:
# Fit XGBoost classifier
from xgboost.spark import SparkXGBClassifier
xgb = SparkXGBClassifier(num_round=10)
xgb_model = xgb.fit(train_df)
xgb_preds = xgb_model.transform(test_df)

In [0]:
from catboost_spark import SparkCatBoostClassifier
catboost = SparkCatBoostClassifier(iterations=10)
catboost_model = catboost.fit(train_df)
catboost_preds = catboost_model.transform(test_df)

In [0]:
# Load US income dataset (e.g., UCI Adult dataset)
income_df = spark.read.csv("/databricks-datasets/adult/adult.data", header=False, inferSchema=True)
columns = [
    "age", "workclass", "fnlwgt", "education", "education_num", "marital_status",
    "occupation", "relationship", "race", "sex", "capital_gain", "capital_loss",
    "hours_per_week", "native_country", "income"
]
income_df = income_df.toDF(*columns)

# Preprocessing: StringIndexer for categorical columns, VectorAssembler for features
from pyspark.ml.feature import StringIndexer, VectorAssembler, OneHotEncoder
categorical_cols = ["workclass", "education", "marital_status", "occupation", "relationship", "race", "sex", "native_country"]
indexers = [StringIndexer(inputCol=col, outputCol=col+"_idx", handleInvalid="keep") for col in categorical_cols]
encoder = OneHotEncoder(inputCols=[col+"_idx" for col in categorical_cols], outputCols=[col+"_enc" for col in categorical_cols])
label_indexer = StringIndexer(inputCol="income", outputCol="label", handleInvalid="keep")
numeric_cols = ["age", "fnlwgt", "education_num", "capital_gain", "capital_loss", "hours_per_week"]
assembler = VectorAssembler(inputCols=[col+"_enc" for col in categorical_cols] + numeric_cols, outputCol="features")

from pyspark.ml import Pipeline
pipeline = Pipeline(stages=indexers + [encoder, label_indexer, assembler])
processed_df = pipeline.fit(income_df).transform(income_df).select("features", "label")

# Split data
train_df, test_df = processed_df.randomSplit([0.8, 0.2], seed=42)

# Fit XGBoost classifier
from xgboost.spark import SparkXGBClassifier
xgb = SparkXGBClassifier(num_round=10)
xgb_model = xgb.fit(train_df)
xgb_preds = xgb_model.transform(test_df)

# Fit CatBoost classifier
from catboost_spark import SparkCatBoostClassifier
catboost = SparkCatBoostClassifier(iterations=10)
catboost_model = catboost.fit(train_df)
catboost_preds = catboost_model.transform(test_df)

# Evaluate models
from pyspark.ml.evaluation import BinaryClassificationEvaluator
evaluator = BinaryClassificationEvaluator(labelCol="label", rawPredictionCol="rawPrediction", metricName="areaUnderROC")
xgb_score = evaluator.evaluate(xgb_preds)
catboost_score = evaluator.evaluate(catboost_preds)

# Publish the best model
if xgb_score >= catboost_score:
    best_model = xgb_model
    best_name = "XGBoost"
else:
    best_model = catboost_model
    best_name = "CatBoost"

# Save/publish the best model
best_model.write().overwrite().save(f"/dbfs/tmp/best_income_classifier_{best_name}")

display(spark.createDataFrame([("XGBoost", xgb_score), ("CatBoost", catboost_score)], ["Model", "AUC"]))